In [1]:
import pandas as pd
import re
import string

from pathlib import Path

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

In [2]:
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package punkt to C:\Users\Shraddha
[nltk_data]     Sharma\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Shraddha
[nltk_data]     Sharma\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Shraddha
[nltk_data]     Sharma\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Shraddha
[nltk_data]     Sharma\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\Shraddha
[nltk_data]     Sharma\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [3]:
DATA_PATH = Path("../data/raw/tickets.csv")

df = pd.read_csv(DATA_PATH)

df.shape

(28587, 16)

In [4]:
english_df = df[
    df["language"].str.lower() == "en"
].copy()

english_df.shape

(16338, 16)

In [6]:
selected_columns = [
    "subject",
    "body",
    "type",
    "queue",
    "priority"
]

english_df = english_df[selected_columns].copy()

english_df.head()

,subject,body,type,queue,priority
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...",Incident,Technical Support,high
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Request,Returns and Exchanges,medium
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",Request,Billing and Payments,low
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Problem,Sales and Pre-Sales,medium
5,Feature Query,"Dear Customer Support,\n\nI hope this message ...",Request,Technical Support,high


In [7]:
english_df[["queue", "priority", "type"]].isna().sum()

queue       0
priority    0
type        0
dtype: int64

In [8]:
english_df["subject"] = (
    english_df["subject"]
    .fillna("")
    .astype(str)
)

english_df["body"] = (
    english_df["body"]
    .fillna("")
    .astype(str)
)

In [9]:
english_df["text_raw"] = (
    english_df["subject"].str.strip()
    + " "
    + english_df["body"].str.strip()
).str.strip()

english_df["text_raw"].head()

1    Account Disruption Dear Customer Support Team,...
2    Query About Smart Home System Integration Feat...
3    Inquiry Regarding Invoice Details Dear Custome...
4    Question About Marketing Agency Software Compa...
5    Feature Query Dear Customer Support,\n\nI hope...
Name: text_raw, dtype: object

In [10]:
empty_text = english_df["text_raw"].str.strip().eq("")

empty_text.sum()

english_df = english_df[~empty_text].copy()

english_df.shape

(16338, 6)

In [11]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\b[\w\.-]+@[\w\.-]+\.\w+\b", " ", text)
    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [12]:
english_df["text_basic_clean"] = (
    english_df["text_raw"].apply(clean_text)
)

english_df[
    ["text_raw", "text_basic_clean"]
].head()

,text_raw,text_basic_clean
1,"Account Disruption Dear Customer Support Team,...",account disruption dear customer support teamn...
2,Query About Smart Home System Integration Feat...,query about smart home system integration feat...
3,Inquiry Regarding Invoice Details Dear Custome...,inquiry regarding invoice details dear custome...
4,Question About Marketing Agency Software Compa...,question about marketing agency software compa...
5,"Feature Query Dear Customer Support,\n\nI hope...",feature query dear customer supportnni hope th...


In [13]:
english_df["text_basic_clean"] = (
    english_df["text_raw"].apply(clean_text)
)

english_df[
    ["text_raw", "text_basic_clean"]
].head()

,text_raw,text_basic_clean
1,"Account Disruption Dear Customer Support Team,...",account disruption dear customer support teamn...
2,Query About Smart Home System Integration Feat...,query about smart home system integration feat...
3,Inquiry Regarding Invoice Details Dear Custome...,inquiry regarding invoice details dear custome...
4,Question About Marketing Agency Software Compa...,question about marketing agency software compa...
5,"Feature Query Dear Customer Support,\n\nI hope...",feature query dear customer supportnni hope th...


In [14]:
english_df["tokens"] = (
    english_df["text_basic_clean"].apply(word_tokenize)
)

english_df[
    ["text_basic_clean", "tokens"]
].head()

,text_basic_clean,tokens
1,account disruption dear customer support teamn...,"[account, disruption, dear, customer, support,..."
2,query about smart home system integration feat...,"[query, about, smart, home, system, integratio..."
3,inquiry regarding invoice details dear custome...,"[inquiry, regarding, invoice, details, dear, c..."
4,question about marketing agency software compa...,"[question, about, marketing, agency, software,..."
5,feature query dear customer supportnni hope th...,"[feature, query, dear, customer, supportnni, h..."


In [15]:
stop_words = set(
    stopwords.words("english")
)

In [16]:
def remove_stopwords(tokens):
    return [
        token
        for token in tokens
        if token not in stop_words
    ]


english_df["tokens_no_stopwords"] = (
    english_df["tokens"].apply(remove_stopwords)
)

english_df[
    ["tokens", "tokens_no_stopwords"]
].head()

,tokens,tokens_no_stopwords
1,"[account, disruption, dear, customer, support,...","[account, disruption, dear, customer, support,..."
2,"[query, about, smart, home, system, integratio...","[query, smart, home, system, integration, feat..."
3,"[inquiry, regarding, invoice, details, dear, c...","[inquiry, regarding, invoice, details, dear, c..."
4,"[question, about, marketing, agency, software,...","[question, marketing, agency, software, compat..."
5,"[feature, query, dear, customer, supportnni, h...","[feature, query, dear, customer, supportnni, h..."


In [17]:
lemmatizer = WordNetLemmatizer()


def lemmatize_tokens(tokens):
    return [
        lemmatizer.lemmatize(token)
        for token in tokens
    ]


english_df["tokens_lemma"] = (
    english_df["tokens_no_stopwords"]
    .apply(lemmatize_tokens)
)

english_df[
    ["tokens_no_stopwords", "tokens_lemma"]
].head()

,tokens_no_stopwords,tokens_lemma
1,"[account, disruption, dear, customer, support,...","[account, disruption, dear, customer, support,..."
2,"[query, smart, home, system, integration, feat...","[query, smart, home, system, integration, feat..."
3,"[inquiry, regarding, invoice, details, dear, c...","[inquiry, regarding, invoice, detail, dear, cu..."
4,"[question, marketing, agency, software, compat...","[question, marketing, agency, software, compat..."
5,"[feature, query, dear, customer, supportnni, h...","[feature, query, dear, customer, supportnni, h..."


In [18]:
english_df["text_clean"] = (
    english_df["tokens_lemma"]
    .apply(lambda tokens: " ".join(tokens))
)

english_df[
    ["text_raw", "text_basic_clean", "text_clean"]
].head

<bound method NDFrame.head of                                                 text_raw  \
1      Account Disruption Dear Customer Support Team,...   
2      Query About Smart Home System Integration Feat...   
3      Inquiry Regarding Invoice Details Dear Custome...   
4      Question About Marketing Agency Software Compa...   
5      Feature Query Dear Customer Support,\n\nI hope...   
...                                                  ...   
28578  Problem with Billing Adjustment An unexpected ...   
28580  Urgent: Incident Involving Data Breach in Medi...   
28582  Performance Problem with Data Analytics Tool T...   
28585  Update Request for SaaS Platform Integration F...   
28586  Inquiry About Project Management Features Look...   

                                        text_basic_clean  \
1      account disruption dear customer support teamn...   
2      query about smart home system integration feat...   
3      inquiry regarding invoice details dear custome...   
4      qu

In [19]:
english_df["raw_word_count"] = (
    english_df["text_raw"]
    .str.split()
    .str.len()
)

english_df["clean_word_count"] = (
    english_df["text_clean"]
    .str.split()
    .str.len()
)

english_df[
    ["raw_word_count", "clean_word_count"]
].describe()

,raw_word_count,clean_word_count
count,16338.000000,16338.000000
mean,58.643041,37.743604
std,26.868554,16.042098
min,2.000000,2.000000
25%,36.000000,24.000000
50%,60.000000,39.000000
75%,82.000000,52.000000
max,172.000000,104.000000


In [20]:
english_df[
    [
        "text_raw",
        "text_basic_clean",
        "text_clean"
    ]
].sample(10, random_state=42)

,text_raw,text_basic_clean,text_clean
25705,Query on Data Analytics Tools for Investment O...,query on data analytics tools for investment o...,query data analytics tool investment optimizat...
17458,Problems with Connection Customers are facing ...,problems with connection customers are facing ...,problem connection customer facing occasional ...
22023,"Hello Customer Support, I am inquiring about o...",hello customer support i am inquiring about op...,hello customer support inquiring optimizing in...
25800,Found Issues with Secure Data Access in Hospit...,found issues with secure data access in hospit...,found issue secure data access hospital system...
25239,Could you offer assistance on securing medical...,could you offer assistance on securing medical...,could offer assistance securing medical data b...
24919,Review and Update Compatibility Settings for E...,review and update compatibility settings for e...,review update compatibility setting enhanced i...
11633,Ensuring Medical Data Security in Healthcare I...,ensuring medical data security in healthcare i...,ensuring medical data security healthcare inqu...
23015,Query Regarding Digital Strategies in the Digi...,query regarding digital strategies in the digi...,query regarding digital strategy digital realm...
22338,Support Required for Hadoop Integration Troubl...,support required for hadoop integration troubl...,support required hadoop integration troublesho...
24273,"encountered a data breach in hospital systems,...",encountered a data breach in hospital systems ...,encountered data breach hospital system result...


In [21]:
english_df["text_clean"].str.strip().eq("").sum()

np.int64(0)

In [22]:
pd.Series({
    "duplicate_raw_text": english_df["text_raw"].duplicated().sum(),
    "duplicate_clean_text": english_df["text_clean"].duplicated().sum()
})

duplicate_raw_text       0
duplicate_clean_text    34
dtype: int64

In [23]:
english_df["queue"].value_counts()
english_df["priority"].value_counts()

priority
medium    6618
high      6346
low       3374
Name: count, dtype: int64

In [24]:
queue_balance = (
    english_df["queue"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .to_frame("percentage")
)

priority_balance = (
    english_df["priority"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .to_frame("percentage")
)

queue_balance, priority_balance

(                                 percentage
 queue                                      
 Technical Support                     28.99
 Product Support                       18.81
 Customer Service                      14.75
 IT Support                            11.89
 Billing and Payments                   9.76
 Returns and Exchanges                  5.02
 Service Outages and Maintenance        4.06
 Sales and Pre-Sales                    3.14
 Human Resources                        2.13
 General Inquiry                        1.44,
           percentage
 priority            
 medium         40.51
 high           38.84
 low            20.65)

In [25]:
model_df = english_df[
    [
        "text_raw",
        "text_basic_clean",
        "text_clean",
        "type",
        "queue",
        "priority"
    ]
].copy()

model_df.head()

,text_raw,text_basic_clean,text_clean,type,queue,priority
1,"Account Disruption Dear Customer Support Team,...",account disruption dear customer support teamn...,account disruption dear customer support teamn...,Incident,Technical Support,high
2,Query About Smart Home System Integration Feat...,query about smart home system integration feat...,query smart home system integration feature de...,Request,Returns and Exchanges,medium
3,Inquiry Regarding Invoice Details Dear Custome...,inquiry regarding invoice details dear custome...,inquiry regarding invoice detail dear customer...,Request,Billing and Payments,low
4,Question About Marketing Agency Software Compa...,question about marketing agency software compa...,question marketing agency software compatibili...,Problem,Sales and Pre-Sales,medium
5,"Feature Query Dear Customer Support,\n\nI hope...",feature query dear customer supportnni hope th...,feature query dear customer supportnni hope me...,Request,Technical Support,high


In [26]:
OUTPUT_PATH = Path(
    "../data/processed/english_tickets_processed.csv"
)

model_df.to_csv(
    OUTPUT_PATH,
    index=False
)

model_df.shape

(16338, 6)